# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides an example for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant JSON-LD schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we inspect the record sets defined in the dataset, their IDs, and the fields they provide.

In [ ]:
# Extracting all RecordSets and summarizing their fields by @id
record_sets = []
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    record_sets = metadata.record_sets
else:
    # Try to infer record sets from metadata if not directly available
    # For croissant datasets, dataset.metadata.record_sets might not be present, but there's usually a method to get available record_sets
    record_sets = dataset.record_sets()

record_set_ids = []
for rset in record_sets:
    # Record set object or dict
    rset_id = getattr(rset, '@id', None) or (rset.get('@id') if isinstance(rset, dict) else None)
    rset_name = getattr(rset, 'name', None) or (rset.get('name') if isinstance(rset, dict) else None)
    if rset_id:
        print(f"RecordSet: {rset_name} | @id: {rset_id}")
        print("  Fields:")
        fields = getattr(rset, 'fields', None) or (rset.get('fields') if isinstance(rset, dict) else None)
        for field in fields or []:
            fid = getattr(field, '@id', None) or (field.get('@id') if isinstance(field, dict) else None)
            fname = getattr(field, 'name', None) or (field.get('name') if isinstance(field, dict) else None)
            print(f"    - {fname} (field @id: {fid})")
        record_set_ids.append(rset_id)
    else:
        # Print warning
        print("No @id found for record set", rset)
        continue

print(f"\nDetected record sets by @id:\n{record_set_ids}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# We extract all records from all detected record sets, using their `@id` as keys for DataFrame objects
dataframes = {}

for rset_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rset_id))
        dataframes[rset_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from RecordSet @id: {rset_id}")
        # Print preview of columns
        print(f"Columns: {dataframes[rset_id].columns.tolist()}")
        print(dataframes[rset_id].head(2))
        print("")
    except Exception as e:
        print(f"Failed to load records for {rset_id}: {e}\n")

# As an example, select the first RecordSet for analysis
main_rs_id = record_set_ids[0] if record_set_ids else None
if main_rs_id:
    print(f"Main record set selected for analysis: {main_rs_id}")
    print(f"Field columns: {dataframes[main_rs_id].columns.tolist()}")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Identify a numeric field by @id (update the field if necessary based on printed columns above)
# Here, we try a likely numeric field; if not, adapt accordingly
numeric_field_candidates = [
    col for col in dataframes[main_rs_id].columns if 'age' in col.lower() or 'interval' in col.lower() or 'number' in col.lower() or 'count' in col.lower()
]
if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]
else:
    numeric_field = dataframes[main_rs_id].select_dtypes(include=['number']).columns[0]

print(f"Numeric field chosen: {numeric_field}")

# Filtering: Example threshold
threshold = 50
df = dataframes[main_rs_id].copy()
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a likely categorical field (e.g., sex, biomarker, or anatomical location)
group_field_candidates = [
    col for col in df.columns if any(x in col.lower() for x in ['sex', 'biomarker', 'location', 'msi', 'metastasis'])
]
if group_field_candidates:
    group_field = group_field_candidates[0]
else:
    group_field = df.select_dtypes(include=['object']).columns[0]

if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame('mean_'+numeric_field)
    print(f"Grouped data by {group_field} (mean of {numeric_field}):")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field].dropna(), bins=20, kde=True)
plt.title(f"Distribution of {numeric_field} (filtered > {threshold})")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

if group_field in filtered_df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully accessed dataset metadata and loaded record sets using their `@id` values.
- Demonstrated data filtering, normalization, and grouping using dynamically identified fields by their `@id`.
- Visualized distributions and group effects for a quantitative field.
- All entities (record set, field, etc.) referenced by their `@id` for reproducibility and schema consistency.

Explore further by adapting groupings or visualizations to specific hypotheses of interest using the `mlcroissant` interface.